# Gradio — Area Coverage, Traffic Counter, PPAC

Jalankan notebook dengan Internet dan GPU aktif. Aktifkan Secret **LocatioIntelligent** pada pengaturan notebook. Isi delapan slot video weekday/weekend × pagi/siang/sore/malam melalui Gradio. Dataset T7SL juga tersedia sebagai contoh.

Notebook menyiapkan aplikasi lalu mencetak **GRADIO_PUBLIC_URL**. Buka URL itu untuk mengisi lokasi/input bisnis, mengunggah video, meninjau geometri/hari/sesi setiap video, memeriksa kesiapan, dan menekan **Mulai proses**. Setup aplikasi tidak menjalankan engine. URL aktif selama sesi notebook/server berjalan. Hentikan cell terakhir untuk menutup UI.

Hasil job menggunakan GCS **aretfak-location-intelligence** dan BQ **locdeproject.location_intelligence**. Siapkan tabel sekali sesuai panduan CLOUD_STORAGE_SETUP.md; model/PBF tetap memakai sumber terpisah yang sudah ada. Folder job dibuat otomatis saat penyimpanan. Jika setup BQ sudah dilakukan, jalankan ulang SQL terbaru untuk menambahkan referensi peta (tanpa menghapus data). Panel hasil membuka tabel BQ dan peta Folium/Esri dari artefak GCS.

Default lokasi Sorong: latitude -0.8994976613947719, longitude 131.3221639428227. Traffic menggunakan kandidat FP32 dan satu video per GPU; akurasi identitas belum tervalidasi. Untuk PPAC dengan delapan video normal, gunakan jam operasional <=15. Operasional >15 jam tetap menunggu sesi tambahan.


In [ ]:
import base64,hashlib,io,zipfile,json,sys,subprocess,importlib,importlib.metadata
from pathlib import Path
BASE=Path('/kaggle/working/gate2-gradio');BASE.mkdir(parents=True,exist_ok=True)
PAYLOAD=base64.b64decode('...')
assert hashlib.sha256(PAYLOAD).hexdigest()=='504d8d462db6d48effc888d2d1ea01e8af711a00320c5d0bd398b9f26141aa0d'
zipfile.ZipFile(io.BytesIO(PAYLOAD)).extractall(BASE)
WHEEL=BASE/'package'/'traffic_location_engines-0.5.1rc1-py3-none-any.whl'
assert hashlib.sha256(WHEEL.read_bytes()).hexdigest()=='f705c2b45b258d3ba62ae0c3c7f40d144060802ca378042c3ab8f5277d6ac730'
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--force-reinstall',str(WHEEL)],check=True)
subprocess.run([sys.executable,'-m','pip','install','gradio==5.49.1','google-cloud-storage==3.6.0','google-cloud-bigquery==3.46.2','google-crc32c==1.7.1','jsonschema==4.26.0','numpy==2.1.3','pandas==2.3.3'],check=True)
for key in list(sys.modules):
    if key=='traffic_location' or key.startswith('traffic_location.'):del sys.modules[key]
importlib.invalidate_caches()
import traffic_location,gradio
assert gradio.__version__=='5.49.1','Restart sesi agar Gradio sesuai versi terpasang.'
package_root=Path(traffic_location.__file__).parent
with zipfile.ZipFile(WHEEL) as package:
    for name in package.namelist():
        if name.startswith('traffic_location/') and name.endswith('.py'):
            assert (package_root/Path(name).relative_to('traffic_location')).read_bytes()==package.read(name),name
print('Paket dan UI siap. Belum ada engine berjalan.')


In [ ]:
from traffic_location.integration.gradio_app import launch
if 'APP' in globals():
    APP.close()
APP, PUBLIC_URL=launch(BASE/'system',BASE/'config',share=True)
print('GRADIO_PUBLIC_URL='+PUBLIC_URL,flush=True)
print('Buka URL publik tersebut. Engine berjalan setelah tombol Mulai proses ditekan.',flush=True)
try:
    APP.block_thread()
finally:
    APP.close()
